# 02. Baseline Model — Logistic Regression (`logreg-v0.0`)

Serves as the Day-1/Day-2 baseline for pipeline validation, interpretable coefficients, and benchmarking against tree-based models.

In [ ]:
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, roc_curve, auc, RocCurveDisplay
import sys

sys.path.append('..')
from training.features import build_features

# Load baseline bundle
bundle_path = Path('../models/logreg-v0.0.pkl')
bundle = joblib.load(bundle_path)
pipeline = bundle['model']
print(f"Loaded baseline version: {bundle['version']}")
print("Saved metrics:", json.dumps(bundle['metrics'], indent=2))

### 1. Extract Interpretable Coefficients

In [ ]:
preprocessor = pipeline.named_steps['preprocessor']
clf = pipeline.named_steps['classifier']

# Handle CalibratedClassifierCV wrapper if calibrated
if hasattr(clf, 'calibrated_classifiers_'):
    base_lr = clf.calibrated_classifiers_[0].estimator
else:
    base_lr = clf

try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    feature_names = ['deal_value', 'days_in_stage', 'num_interactions', 'avg_response_min', 'days_since_last', 'sentiment_trend']

coefs = base_lr.coef_[0]
coef_df = pd.DataFrame({'feature': feature_names[:len(coefs)], 'coef': coefs}).sort_values(by='coef')

plt.figure(figsize=(8, 5))
plt.barh(coef_df['feature'], coef_df['coef'], color=['red' if c < 0 else 'green' for c in coef_df['coef']])
plt.title('Logistic Regression Feature Coefficients')
plt.xlabel('Coefficient Value (Log-Odds Impact)')
plt.axvline(0, color='black', linestyle='--')
plt.tight_layout()
plt.show()